In [4]:
import os


# ── limit for future issues so that code won't break
MAX_FILE_SIZE_BYTES = 10 * 1024 * 1024   # 10 MB hard limit
MAX_SHIFT_VALUE     = 1000               # upper bound for shift inputs
MIN_SHIFT_VALUE     = 0                  # shifts must be non-negative


# Core cipher logic (one character at a time)

def _shift_char(char, amount, base_ord):
    position     = ord(char) - base_ord       # convert to 0-25 index
    new_position = (position + amount) % 26   # shift and wrap
    return chr(new_position + base_ord)       # back to character


# this encrypts the character and null check is added
def encrypt_char(char, shift1, shift2):
    if char is None or not isinstance(char, str) or len(char) != 1:
        return char

    if char.islower():
        return _shift_char(char, shift1 + shift2, ord('a'))
    elif char.isupper():
        return _shift_char(char, shift1 + shift2, ord('A'))
    else:
        return char

def decrypt_char(char, shift1, shift2):
    if char is None or not isinstance(char, str) or len(char) != 1:
        return char

    if char.islower():
        return _shift_char(char, -(shift1 + shift2), ord('a'))
    elif char.isupper():
        return _shift_char(char, -(shift1 + shift2), ord('A'))
    else:
        return char


# File I/O helpers with safety checks

def read_file(file_path):
    # check empty path
    if not file_path:
        print("ERROR: empty file path")
        return None

    try:
        if not os.path.isfile(file_path):
            print(f"ERROR: File not found -> '{file_path}'")
            return None

        size = os.path.getsize(file_path)

        if size == 0:
            print("ERROR: file is empty")
            return None

        if size > MAX_FILE_SIZE_BYTES:
            print("ERROR: file too large")
            return None

        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()

        return content

    except Exception as e:
        print(f"ERROR reading file: {e}")
        return None


def write_file(file_path, content):
    if not file_path or content is None:
        print("ERROR: invalid file or content")
        return False

    try:
        parent = os.path.dirname(file_path)
        if parent and not os.path.exists(parent):
            os.makedirs(parent, exist_ok=True)

        with open(file_path, 'w', encoding='utf-8') as f:
            f.write(content)

        return True

    except Exception as e:
        print(f"ERROR writing file: {e}")
        return False


# encrypt full file

def encrypt(input_path, output_path, shift1, shift2):
    print(f"\n[ENCRYPT] Reading -> {input_path}")

    raw_text = read_file(input_path)
    if raw_text is None:
        return None

    result = []
    for ch in raw_text:
        enc = encrypt_char(ch, shift1, shift2)
        result.append(enc if enc is not None else ch)

    encrypted_text = ''.join(result)

    print(f"[ENCRYPT] Writing -> {output_path}")
    if not write_file(output_path, encrypted_text):
        return None

    print("[ENCRYPT] Done")
    return encrypted_text


# decrypt full file

def decrypt(input_path, output_path, shift1, shift2):
    print(f"\n[DECRYPT] Reading -> {input_path}")

    text = read_file(input_path)
    if text is None:
        return None

    result = []
    for ch in text:
        dec = decrypt_char(ch, shift1, shift2)
        result.append(dec if dec is not None else ch)

    decrypted_text = ''.join(result)

    print(f"[DECRYPT] Writing -> {output_path}")
    if not write_file(output_path, decrypted_text):
        return None

    print("[DECRYPT] Done")
    return decrypted_text


# verify original and decrypted files

def verify(original_path, decrypted_path):
    print(f"\n[VERIFY] Original -> {original_path}")
    print(f"[VERIFY] Decrypted -> {decrypted_path}")

    original  = read_file(original_path)
    decrypted = read_file(decrypted_path)

    if original is None or decrypted is None:
        print("Verification failed")
        return False

    if original == decrypted:
        print("SUCCESS: Files match")
        return True

    print("FAILED: Files do not match")
    return False


# user input

def get_shift_inputs():
    shifts = []

    for name in ("shift1", "shift2"):
        while True:
            user_input = input(f"Enter {name} (0-1000): ").strip()

            if not user_input:
                print("Cannot be empty")
                continue

            try:
                value = int(user_input)

                if value < MIN_SHIFT_VALUE or value > MAX_SHIFT_VALUE:
                    print("Out of range")
                    continue

                shifts.append(value)
                break

            except ValueError:
                print("Invalid number")

    return shifts[0], shifts[1]


# main function (Jupyter compatible)

def main():
    script_dir = os.getcwd()   # FIXED for Jupyter

    raw_path = os.path.join(script_dir, "file", "raw_text.txt")
    encrypted_path = os.path.join(script_dir, "file", "encrypted_text.txt")
    decrypted_path = os.path.join(script_dir, "file", "decrypted_text.txt")

    shift1, shift2 = get_shift_inputs()

    encrypted = encrypt(raw_path, encrypted_path, shift1, shift2)
    if encrypted is None:
        return

    decrypted = decrypt(encrypted_path, decrypted_path, shift1, shift2)
    if decrypted is None:
        return

    verify(raw_path, decrypted_path)


# run

main()


[ENCRYPT] Reading -> C:\Users\ACER\assignment 2\assingment_2\file\raw_text.txt
[ENCRYPT] Writing -> C:\Users\ACER\assignment 2\assingment_2\file\encrypted_text.txt
[ENCRYPT] Done

[DECRYPT] Reading -> C:\Users\ACER\assignment 2\assingment_2\file\encrypted_text.txt
[DECRYPT] Writing -> C:\Users\ACER\assignment 2\assingment_2\file\decrypted_text.txt
[DECRYPT] Done

[VERIFY] Original -> C:\Users\ACER\assignment 2\assingment_2\file\raw_text.txt
[VERIFY] Decrypted -> C:\Users\ACER\assignment 2\assingment_2\file\decrypted_text.txt
SUCCESS: Files match
